In [ ]:
https://colab.research.google.com/drive/1v3iqmoHxvcihzNlysVmVlmAqkZy8T3tc#scrollTo=EHQp85RffhMW

この内容をローカルで実行したい。
ただし、テストはメタデータフィルターでやりたいので以下を良い感じに流用する。


SAMPLE_TEXTS_WITH_METADATA = [
    {
        "text": (
            "今日は素晴らしい一日でした。朝に近所の公園を散歩し、桜が満開で癒されました。"
        ),
        "metadata": {
            "full_doc_id": "doc-park",
            "chunk_order_index": 0,
            "topic": "outdoor",
            "mood": "relaxed",
            "keywords": ["散歩", "公園"],
        },
    },
    {
        "text": (
            "仕事で取り組んでいたAIプロジェクトが完了し、チーム全員で大きな達成感を味わいました。"
        ),
        "metadata": {
            "full_doc_id": "doc-project",
            "chunk_order_index": 1,
            "topic": "work",
            "mood": "達成",
            "keywords": ["仕事", "プロジェクト"],
        },
    },
    {
        "text": (
            "週末に初めてパスタを一から作り、苦労しながらもコクのあるカルボナーラが完成しました。"
        ),
        "metadata": {
            "full_doc_id": "doc-cooking",
            "chunk_order_index": 2,
            "topic": "cooking",
            "mood": "挑戦",
            "keywords": ["料理", "パスタ"],
        },
    },
    {
        "text": (
            "村上春樹の小説を読み進めながら、深層学習の技術書で理論も学んでいます。"
        ),
        "metadata": {
            "full_doc_id": "doc-reading",
            "chunk_order_index": 3,
            "topic": "reading",
            "mood": "集中",
            "keywords": ["読書", "本"],
        },
    },
    {
        "text": (
            "友人と映画館で『君の名は。』を鑑賞し、感動的なストーリーに胸が熱くなりました。"
        ),
        "metadata": {
            "full_doc_id": "doc-movie",
            "chunk_order_index": 4,
            "topic": "movie",
            "mood": "感動",
            "keywords": ["映画"],
        },
    },
]


async def _sample_insert_texts(rag_instance: _SampleRAG):
    for item in SAMPLE_TEXTS_WITH_METADATA:
        await rag_instance.ainsert(item["text"].strip(), metadata=item["metadata"])


async def _prepare_sample_rag() -> _SampleRAG:
    rag = _SampleRAG()
    await _sample_insert_texts(rag)
    return rag


async def _run_sample_queries(
    rag_instance: _SampleRAG,
    queries: list[str],
    metadata_filters: dict[str, str],
):
    modes = ["naive", "mini", "light"]
    results: dict[str, dict[str, list[str]]] = {}

    for query in queries:
        mode_results: dict[str, list[str]] = {}
        for mode in modes:
            answer = await rag_instance.aquery(
                query,
                param=QueryParam(
                    mode=mode,
                    metadata_filters=metadata_filters,
                    only_need_context=True,
                ),
            )
            mode_results[mode] = answer
        results[query] = mode_results

    return results


def test_sample_rag_filters_by_full_doc_id():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"full_doc_id": "doc-movie"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("映画について教えて", param=param))

    assert len(results) == 1
    assert "映画" in results[0]


def test_sample_rag_filters_by_multiple_conditions():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"topic": "cooking", "mood": "挑戦"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("料理で何を作りましたか？", param=param))

    assert len(results) == 1
    assert "カルボナーラ" in results[0]


def test_sample_rag_filters_accept_stringified_numbers():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"chunk_order_index": "2"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("料理で何を作りましたか？", param=param))

    assert len(results) == 1
    assert "パスタ" in results[0]


def test_sample_rag_filters_no_match_returns_empty_list():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"topic": "movie", "mood": "集中"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("読んでいる本について教えて", param=param))

    assert results == []


def test_run_sample_queries_applies_filters_to_each_mode():
    rag = asyncio.run(_prepare_sample_rag())
    metadata_filters = {"topic": "movie"}
    queries = [
        "映画について教えて",
        "散歩について詳しく教えて",
    ]

    results = asyncio.run(_run_sample_queries(rag, queries, metadata_filters))


In [33]:
!nvidia-smi

Fri Oct 24 12:49:20 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 528.24       Driver Version: 528.24       CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name            TCC/WDDM | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ... WDDM  | 00000000:01:00.0  On |                  Off |
|  0%   36C    P2    60W / 450W |    681MiB / 24564MiB |      2%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [2]:
!uv run python --version

Python 3.12.11


In [ ]:
# インストールしていく
!uv add pip

In [18]:
!uv add python-dotenv json_repair rouge numpy pandas tiktoken nltk pipmaster

Resolved 175 packages in 775ms
Prepared 3 packages in 1.48s
Uninstalled 1 package in 1ms
Installed 3 packages in 10ms
 + ascii-colors==0.11.4
 ~ minirag-hku==0.0.2 (from file:///C:/Users/kbpsh/OneDrive/development/project/minirag_dayo)
 + pipmaster==1.0.9


In [16]:
!uv add sentence_transformers

Resolved 173 packages in 1.03s
Prepared 13 packages in 25.02s
Uninstalled 1 package in 2ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 16 packages in 8.43s
 + filelock==3.20.0
 + fsspec==2025.9.0
 + huggingface-hub==0.36.0
 ~ minirag-hku==0.0.2 (from file:///C:/Users/kbpsh/OneDrive/development/project/minirag_dayo)
 + mpmath==1.3.0
 + networkx==3.5
 + pillow==12.0.0
 + safetensors==0.6.2
 + scikit-learn==1.7.2
 + scipy==1.16.2
 + sentence-transformers==5.1.2
 + sympy==1.14.0
 + threadpoolctl==3.6.0
 + tokenizers==0.22.1
 + torch==2.9.0
 + transformers==4.57.1


In [3]:
!uv add openai tenacity protobuf sentencepiece

Resolved 180 packages in 485ms
Prepared 2 packages in 1.22s
Uninstalled 1 package in 2ms
Installed 2 packages in 10ms
 ~ minirag-hku==0.0.2 (from file:///C:/Users/kbpsh/OneDrive/development/project/minirag_dayo)
 + sentencepiece==0.2.1


In [1]:
# GPUを認識させる
# 現在のPyTorchを完全にアンインストール
!uv run python -m pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.9.0
Uninstalling torch-2.9.0:
  Successfully uninstalled torch-2.9.0
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121


Uninstalled 2 packages in 2.53s
Installed 1 package in 10.32s


In [2]:
# CUDA版を再インストール
!uv run python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-win_amd64.whl (6.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (4.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (2449.3 MB)
  Using cached https://download.pytorch.org/whl/sympy-1.13.1-py3-none-any.whl (6.2 MB)

  Attempting uninstall: sympy

    Found existing installation: sympy 1.14.0

   ---------------------------------------- 0/4 [sympy]
    Uninstalling sympy-1.14.0:
   ---------------------------------------- 0/4 [sympy]
   ---------------------------------------- 0/4 [sympy]
   ---------------------------------------- 0/4 [sympy]
      Successfully uninstalled sympy-1.14.0
   ---------------------------------------- 0/4 [sympy]
   ---------------------------------------- 0/4 [sympy]
   -------------------

Installed 1 package in 14.83s
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
minirag-hku 0.0.2 requires torch[cuda]>=2.8.0, but you have torch 2.5.1+cu121 which is incompatible.


In [ ]:
!uv lock --upgrade

In [ ]:
!uv sync

In [6]:
!python -c "import sys, torch; print('python exe:', sys.executable); print('python version:', sys.version.splitlines()[0]); print('torch.__version__:', torch.__version__); print('torch.version.cuda:', torch.version.cuda); print('torch.cuda.is_available():', torch.cuda.is_available())"

python exe: C:\Users\kbpsh\OneDrive\development\project\minirag_dayo\.venv\Scripts\python.exe
python version: 3.12.11 (main, Jun 26 2025, 21:17:44) [MSC v.1944 64 bit (AMD64)]
torch.__version__: 2.5.1+cu121
torch.version.cuda: 12.1
torch.cuda.is_available(): True


In [1]:
# GPU を反映させるには一旦Jupyter Notebookサーバーを再起動させる必要あり
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device count:', torch.cuda.device_count())
    print('Current device:', torch.cuda.current_device())
    print('Device name:', torch.cuda.get_device_name(torch.cuda.current_device()))

PyTorch version: 2.5.1+cu121
CUDA available: True
Device count: 1
Current device: 0
Device name: NVIDIA GeForce RTX 4090


In [4]:
# feature-metadata-filtering ブランチでOK
!git branch

* feature-metadata-filtering
  main
  metadata-filter


In [10]:
# 必要なライブラリのインポート
import os
import tempfile
from minirag import MiniRAG, QueryParam
from minirag.llm.hf import (
    hf_model_complete,
    hf_embed,
)
# from minirag.llm.openai import openrouter_openai_complete
from minirag.llm.openai import openai_complete_if_cache
from minirag.utils import EmbeddingFunc
from minirag.utils import (
    wrap_embedding_func_with_attrs,
    locate_json_string_body_from_string,
    safe_unicode_decode,
    logger,
)
from transformers import AutoModel, AutoTokenizer
import asyncio
import warnings
warnings.filterwarnings('ignore')

## 環境変数

In [11]:
import os
from dotenv import load_dotenv

# .envファイルを読み込む
load_dotenv()

# 環境変数を取得する
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
HF_TOKEN = os.getenv('HF_TOKEN')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')


os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ["OPENAI_API_KEY"] = OPENROUTER_API_KEY


print(f"GEMINI_API_KEY: {GEMINI_API_KEY[:5]}********************")
print(f"HF_TOKEN: {HF_TOKEN[:5]}********************")
print(f"OPENROUTER_API_KEY: {OPENROUTER_API_KEY[:5]}********************")

GEMINI_API_KEY: AIzaS********************
HF_TOKEN: hf_UK********************
OPENROUTER_API_KEY: sk-or********************


## セッティング

日本語用埋め込みモデル

In [2]:
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

# Download from the 🤗 Hub
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("cl-nagoya/ruri-v3-30m", device=device)

# Ruri v3 employs a 1+3 prefix scheme to distinguish between different types of text inputs:
# "" (empty string) is used for encoding semantic meaning.
# "トピック: " is used for classification, clustering, and encoding topical information.
# "検索クエリ: " is used for queries in retrieval tasks.
# "検索文書: " is used for documents to be retrieved.
sentences = [
    "川べりでサーフボードを持った人たちがいます",
    "サーファーたちが川べりに立っています",
    "トピック: 瑠璃色のサーファー",
    "検索クエリ: 瑠璃色はどんな色？",
    "検索文書: 瑠璃色（るりいろ）は、紫みを帯びた濃い青。名は、半貴石の瑠璃（ラピスラズリ、英: lapis lazuli）による。JIS慣用色名では「こい紫みの青」（略号 dp-pB）と定義している[1][2]。",
]

embeddings = model.encode(sentences, convert_to_tensor=True)
print(embeddings.size())
# [5, 256]

similarities = F.cosine_similarity(embeddings.unsqueeze(0), embeddings.unsqueeze(1), dim=2)
print(similarities)
# [[1.0000, 0.9540, 0.8512, 0.7322, 0.7274],
#  [0.9540, 1.0000, 0.8531, 0.7437, 0.7305],
#  [0.8512, 0.8531, 1.0000, 0.8910, 0.8649],
#  [0.7322, 0.7437, 0.8910, 1.0000, 0.9479],
#  [0.7274, 0.7305, 0.8649, 0.9479, 1.0000]]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

C:\Users\kbpsh\OneDrive\development\project\minirag_dayo\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kbpsh\.cache\huggingface\hub\models--cl-nagoya--ruri-v3-30m. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


torch.Size([5, 256])
tensor([[1.0000, 0.9539, 0.8513, 0.7321, 0.7273],
        [0.9539, 1.0000, 0.8531, 0.7436, 0.7305],
        [0.8513, 0.8531, 1.0000, 0.8908, 0.8647],
        [0.7321, 0.7436, 0.8908, 1.0000, 0.9477],
        [0.7273, 0.7305, 0.8647, 0.9477, 1.0000]], device='cuda:0')


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [32]:
# 埋め込みモデルの設定
EMBEDDING_MODEL = "cl-nagoya/ruri-v3-30m"
# LLMの設定
# LLM_MODEL = "Qwen/Qwen3-1.7B"  # または "Qwen/Qwen3-4B", "Qwen/Qwen3-1.7B" など
# LLM_MODEL = "jaeyong2/Qwen2.5-3B-Instruct-Ja-SFT"
# LLM_MODEL = "qwen/qwen3-235b-a22b:free"         # 精度が足りなくてJSONのパースで失敗する
LLM_MODEL = "qwen/qwen3-30b-a3b-instruct-2507"



# 作業ディレクトリの作成
WORKING_DIR = "/tmp/minirag_demo"
os.makedirs(WORKING_DIR, exist_ok=True)

print(f"作業ディレクトリ: {WORKING_DIR}")


# DATA_PATH = args.datapath
# QUERY_PATH = args.querypath
# OUTPUT_PATH = args.outputpath
# print("USING LLM:", LLM_MODEL)
# print("USING WORKING DIR:", WORKING_DIR)

作業ディレクトリ: /tmp/minirag_demo


In [33]:
async def openrouter_openai_complete(
    prompt,
    system_prompt=None,
    history_messages=[],
    keyword_extraction=False,
    api_key: str = None,
    **kwargs,
) -> str:
    # if api_key:
    #     os.environ["OPENROUTER_API_KEY"] = api_key

    keyword_extraction = kwargs.pop("keyword_extraction", None)
    result = await openai_complete_if_cache(
        LLM_MODEL,  # change accordingly
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
        **kwargs,
    )
    if keyword_extraction:  # TODO: use JSON API
        return locate_json_string_body_from_string(result)
    return result

In [22]:
%pwd

%ls

 ドライブ C のボリューム ラベルは Windows です
 ボリューム シリアル番号は F444-D319 です

 C:\Users\kbpsh\OneDrive\development\project\minirag_dayo のディレクトリ

2025/10/24  18:31    <DIR>          .
2025/10/23  22:48    <DIR>          ..
2025/03/20  09:30               241 .cursorignore
2025/10/24  12:42               202 .env
2025/10/07  00:46             2,088 .gitignore
2025/10/24  12:55    <DIR>          .ipynb_checkpoints
2025/10/21  08:54               483 .pre-commit-config.yaml
2025/10/24  12:24    <DIR>          .venv
2025/10/21  11:32                 0 ★ディレクトリ名は適当。これはMiniRAGをそのままクローンしたのとほぼ同じ.txt
2025/10/21  08:56    <DIR>          assets
2025/10/21  08:54               234 Communication.md
2025/10/21  08:56    <DIR>          dataset
2025/10/21  08:54               420 docker-compose.yml
2025/10/21  08:54             1,210 Dockerfile
2025/10/21  09:12    <DIR>          docs
2025/10/21  08:56    <DIR>          graph-visuals
2025/10/21  08:54             1,067 LICENSE
2025/10/24  12:22             3,783 main.py


In [34]:
# MiniRAGインスタンスの作成
rag = MiniRAG(
    working_dir=WORKING_DIR,

    # llm_model_func=hf_model_complete,
    llm_model_func=openrouter_openai_complete,

    llm_model_max_token_size=200,
    llm_model_name=LLM_MODEL,
    embedding_func=EmbeddingFunc(
        embedding_dim=256,
        max_token_size=1000,
        func=lambda texts: hf_embed(
            texts,
            tokenizer=AutoTokenizer.from_pretrained(EMBEDDING_MODEL),
            embed_model=AutoModel.from_pretrained(EMBEDDING_MODEL),
        ),
    ),
)

print("MiniRAGが初期化されました！")

INFO:minirag:Logger initialized for working directory: /tmp/minirag_demo
INFO:minirag:Load KV json_doc_status_storage with 0 data
INFO:minirag:Load KV llm_response_cache with 0 data
INFO:minirag:Load KV full_docs with 1 data
INFO:minirag:Load KV text_chunks with 1 data
INFO:minirag:Loaded graph from /tmp/minirag_demo\graph_chunk_entity_relation.graphml with 25 nodes, 21 edges
INFO:nano-vectordb:Load (22, 256) data
INFO:nano-vectordb:Init {'embedding_dim': 256, 'metric': 'cosine', 'storage_file': '/tmp/minirag_demo\\vdb_entities.json'} 22 data
INFO:nano-vectordb:Load (22, 256) data
INFO:nano-vectordb:Init {'embedding_dim': 256, 'metric': 'cosine', 'storage_file': '/tmp/minirag_demo\\vdb_entities_name.json'} 22 data
INFO:nano-vectordb:Load (21, 256) data
INFO:nano-vectordb:Init {'embedding_dim': 256, 'metric': 'cosine', 'storage_file': '/tmp/minirag_demo\\vdb_relationships.json'} 21 data
INFO:nano-vectordb:Load (1, 256) data
INFO:nano-vectordb:Init {'embedding_dim': 256, 'metric': 'cosin

MiniRAGが初期化されました！


In [35]:
# これをやっておかないと HuggingFace の認証で失敗する

from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL, token=HF_TOKEN)
model = AutoModelForMaskedLM.from_pretrained(EMBEDDING_MODEL, token=HF_TOKEN)

Some weights of ModernBertForMaskedLM were not initialized from the model checkpoint at cl-nagoya/ruri-v3-30m and are newly initialized: ['decoder.bias', 'head.dense.weight', 'head.norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [36]:
# prompt: ロードしたmodelの開放

# メモリ解放のためにモデルを削除
del model
del tokenizer

# PyTorchのキャッシュをクリア (GPUを使用している場合)
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Pythonのガベージコレクションを実行
import gc
gc.collect()

print("モデルとトークナイザーが解放されました。")

モデルとトークナイザーが解放されました。


In [26]:
# 約12分かかった
import time
start_time = time.time()

# サンプルテキストデータ
sample_texts = [
    """
今日は素晴らしい一日でした。朝早く起きて、近所の公園を散歩しました。
桜の花が満開で、とても美しかったです。午後は友人と映画を見に行きました。
「君の名は。」という映画で、とても感動的でした。
夜は家族と一緒に夕食を取り、楽しい時間を過ごしました。
""",
    """
昨日は仕事で大きなプロジェクトが完了しました。
チーム全員で3ヶ月間取り組んできたAIシステムの開発が終わりました。
機械学習モデルの精度が95%を超え、クライアントからも高い評価をいただきました。
今夜はチームメンバーと祝賀会を開く予定です。
""",
    """
週末は料理に挑戦しました。初めてパスタを一から作ってみました。
小麦粉から麺を作るのは思っていたより難しかったですが、
最終的にはとても美味しいカルボナーラができました。
次回はリゾットに挑戦してみたいと思います。
""",
    """
読書が趣味で、最近は村上春樹の「ノルウェイの森」を読んでいます。
主人公の心情描写がとても繊細で、引き込まれます。
また、技術書も読んでおり、「深層学習」について学んでいます。
理論と実践のバランスが取れた良い本だと思います。
"""
]

# データの挿入
print("データを挿入中...")

async def insert_texts(rag_instance, texts):
    for i, text in enumerate(texts):
        print(f"テキスト {i+1}/{len(texts)} を挿入中...")
        await rag_instance.ainsert(text.strip())

    print("\nすべてのデータが挿入されました！")


# イベントループが既に実行中の場合
try:
    await insert_texts(rag, sample_texts)
except RuntimeError:
    # 新しいループで実行
    asyncio.run(insert_texts(rag, sample_texts))

end_time = time.time()
elapsed_time = end_time - start_time
print(f"処理時間: {elapsed_time:.4f}秒")

INFO:minirag:No new unique documents were found.
INFO:minirag:No documents to process
INFO:minirag:Performing entity extraction on newly processed chunks


データを挿入中...
テキスト 1/4 を挿入中...


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


⠙ Processed 1 chunks, 7 entities(duplicated), 5 relations(duplicated)

INFO:minirag:Inserting 7 vectors to entities


...

テキスト 3/4 を挿入中...


Generating embeddings: 100%|███████████████| 1/1 [00:01<00:00,  1.49s/batch]
INFO:minirag:Document processing pipeline completed
INFO:minirag:Performing entity extraction on newly processed chunks
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


JSONDecodeError: Expecting value: line 565 column 1 (char 3102)

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


⠹ Processed 2 chunks, 8 entities(duplicated), 2 relations(duplicated)

In [ ]:
SAMPLE_TEXTS_WITH_METADATA = [
    {
        "text": (
            "今日は素晴らしい一日でした。朝早く起きて、近所の公園を散歩しました。桜の花が満開で、とても美しかったです。午後は友人と映画を見に行きました。「君の名は。」という映画で、とても感動的でした。夜は家族と一緒に夕食を取り、楽しい時間を過ごしました。"
        ),
        "metadata": {
            "full_doc_id": "doc-park",
            "chunk_order_index": 0,
            "topic": "outdoor",
            "mood": "relaxed",
            "keywords": ["散歩", "公園"],
        },
    },
    {
        "text": (
            "昨日は仕事で大きなプロジェクトが完了しました。チーム全員で3ヶ月間取り組んできたAIシステムの開発が終わりました。機械学習モデルの精度が95%を超え、クライアントからも高い評価をいただきました。今夜はチームメンバーと祝賀会を開く予定です。"
        ),
        "metadata": {
            "full_doc_id": "doc-project",
            "chunk_order_index": 1,
            "topic": "work",
            "mood": "達成",
            "keywords": ["仕事", "プロジェクト"],
        },
    },
    {
        "text": (
            "週末は料理に挑戦しました。初めてパスタを一から作ってみました。小麦粉から麺を作るのは思っていたより難しかったですが、最終的にはとても美味しいカルボナーラができました。次回はリゾットに挑戦してみたいと思います。"
        ),
        "metadata": {
            "full_doc_id": "doc-cooking",
            "chunk_order_index": 2,
            "topic": "cooking",
            "mood": "挑戦",
            "keywords": ["料理", "パスタ"],
        },
    },
    {
        "text": (
            "読書が趣味で、最近は村上春樹の「ノルウェイの森」を読んでいます。主人公の心情描写がとても繊細で、引き込まれます。また、技術書も読んでおり、「深層学習」について学んでいます。理論と実践のバランスが取れた良い本だと思います。"
        ),
        "metadata": {
            "full_doc_id": "doc-reading",
            "chunk_order_index": 3,
            "topic": "reading",
            "mood": "集中",
            "keywords": ["読書", "本"],
        },
    },
]


async def _sample_insert_texts(rag_instance):
    for item in SAMPLE_TEXTS_WITH_METADATA:
        # metadatas: dict | list[dict] | None = None
        await rag_instance.ainsert(item["text"].strip(), metadatas=item["metadata"])


async def _prepare_sample_rag():
    await _sample_insert_texts(rag)
    return rag


# メタデータ入りのデータ挿入
await _prepare_sample_rag()

INFO:minirag:No new unique documents were found.
INFO:minirag:No documents to process
INFO:minirag:Performing entity extraction on newly processed chunks
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


⠙ Processed 1 chunks, 8 entities(duplicated), 8 relations(duplicated)

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


⠼ Processed 4 chunks, 38 entities(duplicated), 46 relations(duplicated)

INFO:minirag:Inserting 32 vectors to entities


Generating embeddings: 100%|███████████████| 1/1 [00:01<00:00,  1.38s/batch]
INFO:minirag:Inserting 32 vectors to entities
Generating embeddings: 100%|███████████████| 1/1 [00:01<00:00,  1.63s/batch]
INFO:minirag:Inserting 32 vectors to entities_name
Generating embeddings: 100%|███████████████| 1/1 [00:01<00:00,  1.26s/batch]
INFO:minirag:Inserting 42 vectors to relationships
Generating embeddings: 100%|███████████████| 2/2 [00:02<00:00,  1.33s/batch]
INFO:minirag:Writing graph with 56 nodes, 63 edges
INFO:minirag:Stored 1 new unique documents
INFO:minirag:Number of batches to process: 1
INFO:minirag:Inserting 1 vectors to chunks
Generating embeddings: 100%|███████████████| 1/1 [00:01<00:00,  1.26s/batch]
INFO:minirag:Document processing pipeline completed
INFO:minirag:Performing entity extraction on newly processed chunks
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/complet

In [ ]:
async def _run_sample_queries(
    rag_instance: _SampleRAG,
    queries: list[str],
    metadata_filters: dict[str, str],
):
    modes = ["naive", "mini", "light"]
    results: dict[str, dict[str, list[str]]] = {}

    for query in queries:
        mode_results: dict[str, list[str]] = {}
        for mode in modes:
            answer = await rag_instance.aquery(
                query,
                param=QueryParam(
                    mode=mode,
                    metadata_filters=metadata_filters,
                    only_need_context=True,
                ),
            )
            mode_results[mode] = answer
        results[query] = mode_results

    return results


def test_sample_rag_filters_by_full_doc_id():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"full_doc_id": "doc-movie"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("映画について教えて", param=param))

    assert len(results) == 1
    assert "映画" in results[0]


def test_sample_rag_filters_by_multiple_conditions():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"topic": "cooking", "mood": "挑戦"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("料理で何を作りましたか？", param=param))

    assert len(results) == 1
    assert "カルボナーラ" in results[0]


def test_sample_rag_filters_accept_stringified_numbers():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"chunk_order_index": "2"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("料理で何を作りましたか？", param=param))

    assert len(results) == 1
    assert "パスタ" in results[0]


def test_sample_rag_filters_no_match_returns_empty_list():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"topic": "movie", "mood": "集中"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("読んでいる本について教えて", param=param))

    assert results == []


def test_run_sample_queries_applies_filters_to_each_mode():
    rag = asyncio.run(_prepare_sample_rag())
    metadata_filters = {"topic": "movie"}
    queries = [
        "映画について教えて",
        "散歩について詳しく教えて",
    ]

    results = asyncio.run(_run_sample_queries(rag, queries, metadata_filters))
